In [7]:
import sys
import os

sys.path.insert(0, '/home/ubuntu/Bakalarka/QSPRpred/qsprpred/extra/gpu/models')

import torch

from qsprpred.data import QSPRDataset, RandomSplit
from qsprpred.data.descriptors.fingerprints import MorganFP
import pandas as pd
from qsprpred.data.descriptors.sets import RDKitDescs
from MolEval import MolEmb 
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
import numpy as np
import pandas as pd
from sklearn.feature_selection import VarianceThreshold
from qsprpred.data.descriptors.sets import RDKitDescs

In [8]:
X1 = pd.read_csv("mod_data/X1.1", index_col="QSPRID")
X2 = pd.read_csv("mod_data/X2.1", index_col="QSPRID")
X3 = pd.read_csv("mod_data/X3.1", index_col="QSPRID")
y1 = pd.read_csv("mod_data/y1.1", index_col="QSPRID")
y2 = pd.read_csv("mod_data/y2.1", index_col="QSPRID")
y3 = pd.read_csv("mod_data/y3.1", index_col="QSPRID")


In [9]:
X1.columns = X1.columns.astype(str)
X2.columns = X2.columns.astype(str)
X3.columns = X3.columns.astype(str)

imp_mean = SimpleImputer(missing_values=pd.NA, strategy='mean')
X1 = imp_mean.fit_transform(X1)
X2 = imp_mean.transform(X2)
X3 = imp_mean.transform(X3)
scaler = StandardScaler()
scaler.fit(X1)
X1 = scaler.transform(X1)
X2 = scaler.transform(X2)
X3 = scaler.transform(X3)

In [10]:
from sklearn.decomposition import PCA
pca = PCA(n_components=0.95)
pca.fit_transform(X1)
cum_var = np.cumsum(pca.explained_variance_ratio_)

# Např. najít počet komponent, které vysvětlí 95 % rozptylu
n_components = np.argmax(cum_var >= 0.95) + 1
from sklearn.decomposition import PCA
pca = PCA(n_components=n_components)
X1 = pca.fit_transform(X1)
X2 = pca.transform(X2)
X3 = pca.transform(X3)

In [11]:
n_components

1854

In [12]:
display(X1)

array([[ 1.97359400e+01, -1.80045303e+00, -2.09784283e+01, ...,
         7.37035893e-01,  1.82937991e+00,  9.54246928e-02],
       [ 1.99750253e+01, -1.96940543e+00, -2.10829740e+01, ...,
         6.69776602e-01,  1.55783721e+00, -1.97299370e-02],
       [ 2.32364501e+01, -3.14355004e+00, -2.32346152e+01, ...,
         1.80141566e-01,  6.03913545e-01,  5.87554304e-01],
       ...,
       [-9.63150399e+00, -3.81972300e+00, -3.86791085e+00, ...,
         3.46325191e-01,  5.68542508e-01, -1.07363003e+00],
       [-9.64944375e+00, -3.75505234e+00, -3.80454489e+00, ...,
         3.84167487e-01,  5.05283650e-01, -1.12783638e+00],
       [-6.39316025e+00, -1.33681263e+00, -1.68179756e+00, ...,
        -1.08109255e-01, -8.23467919e-01,  4.82035152e-01]])

In [13]:
from sklearn.svm import SVC

svc = SVC()
svc.fit(X1, y1)

/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


SVC()

In [14]:
pred = svc.predict(X2)

In [15]:
from sklearn.metrics import matthews_corrcoef, accuracy_score, f1_score

print("Acc:", accuracy_score(pred, y2))
print("Mcc:", matthews_corrcoef(pred, y2))

Acc: 0.7296520906479412
Mcc: 0.43319990467225467


In [18]:
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import matthews_corrcoef, accuracy_score, f1_score
from tqdm import tqdm
import pandas as pd
param_grid = ParameterGrid({
    'C': np.logspace(-3, 3, num=7).tolist(), # Např. [0.01, 0.1, 1.0, 10.0, 100.0]
    'kernel': ['linear', 'rbf', 'poly'], # Omezeno na nejčastější kernely
    'gamma': np.logspace(-2, 1, num=4).tolist(), # Např. [0.01, 0.1, 1.0, 10.0] - relevantní pro 'rbf'
})

val_mcc = []
val_f1 = []
val_acc = []
comb_tried = []
for param in tqdm(param_grid, total=len(param_grid)):
    if param['kernel'] == 'linear' and param['C'] in comb_tried:
        val_mcc.append(-2)
        val_f1.append(-1)
        val_acc.append(-1)
        print("Params skipped:", param,)
        pass
    else:
        if param['kernel'] == 'linear':
            comb_tried.append(param['C'])
        svc = SVC(class_weight="balanced",random_state=69, **param)
        svc.fit(X1, y1["Y"])
        pred_svc = svc.predict(X2)
        val_mcc.append(matthews_corrcoef(pred_svc, y2["Y"]))
        val_f1.append(f1_score(pred_svc, y2["Y"]))
        val_acc.append(accuracy_score(pred_svc, y2["Y"]))
        print("Params:", param, "\nMCC:", matthews_corrcoef(pred_svc, y2["Y"]))



  1%|▌                                             | 1/84 [01:17<1:47:15, 77.53s/it]

Params: {'C': 0.001, 'gamma': 0.01, 'kernel': 'linear'} 
MCC: 0.39806367828453676


  2%|█                                            | 2/84 [04:17<3:08:31, 137.95s/it]

Params: {'C': 0.001, 'gamma': 0.01, 'kernel': 'rbf'} 
MCC: 0.0


  4%|█▌                                           | 3/84 [07:54<3:54:40, 173.84s/it]

...

Params: {'C': 1000.0, 'gamma': 1.0, 'kernel': 'poly'} 
MCC: 0.2919970921369111
Params skipped: {'C': 1000.0, 'gamma': 10.0, 'kernel': 'linear'}


 99%|██████████████████████████████████████████▍| 83/84 [10:00:16<08:07, 487.70s/it]

Params: {'C': 1000.0, 'gamma': 10.0, 'kernel': 'rbf'} 
MCC: 0.0


100%|███████████████████████████████████████████| 84/84 [10:01:17<00:00, 429.49s/it]

Params: {'C': 1000.0, 'gamma': 10.0, 'kernel': 'poly'} 
MCC: 0.08785686368934213


In [19]:
import pandas as pd
svc_res = pd.DataFrame(param_grid)

In [20]:
svc_res["mcc"] = val_mcc
svc_res["f1"] = val_f1
svc_res["acc"] = val_acc

In [21]:
svc_res.sort_values(by=["mcc", "f1", "acc"], ascending=False, inplace=True)

In [22]:
svc_res.to_csv("res/svc_without_smote_res_val_1.1.csv")

In [23]:
svc_res

,C,gamma,kernel,mcc,f1,acc
0,0.001,0.01,linear,0.398064,0.754646,0.709224
12,0.010,0.01,linear,0.365669,0.746483,0.695180
59,10.000,10.00,poly,0.322938,0.747612,0.679540
2,0.001,0.01,poly,0.317576,0.768000,0.676029
24,0.100,0.01,linear,0.311388,0.717532,0.666773
...,...,...,...,...,...,...
66,100.000,1.00,linear,-2.000000,-1.000000,-1.000000
69,100.000,10.00,linear,-2.000000,-1.000000,-1.000000
75,1000.000,0.10,linear,-2.000000,-1.000000,-1.000000
78,1000.000,1.00,linear,-2.000000,-1.000000,-1.000000


In [15]:
import numpy as np
best_params = param_grid[np.argmax(val_mcc)]

In [16]:
best_params

{'shrinking': True,
 'kernel': 'sigmoid',
 'gamma': 'scale',
 'degree': 2,
 'coef0': 0.0,
 'C': 1}

In [17]:
svc = SVC(**best_params)
svc.fit(X1, y1)

/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/sklearn/utils/validation.py:1229: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


SVC(C=1, degree=2, kernel='sigmoid')

In [18]:
pred = svc.predict(X2)

print("Acc:", accuracy_score(pred, y2))
print("F1:", f1_score(pred, y2))
print("Mcc:", matthews_corrcoef(pred, y2))

Acc: 0.9704069050554871
F1: 0.9848484848484849
Mcc: 0.44317007044298046


In [19]:
pred_test = svc.predict(X3)

print("Acc:", accuracy_score(pred_test, y3))
print("F1:", f1_score(pred_test, y3))
print("Mcc:", matthews_corrcoef(pred_test, y3))

Acc: 0.9722222222222222
F1: 0.9857988165680474
Mcc: 0.4437119029360156
